In [ ]:

import torch

TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda

print("Torch:", TORCH_VERSION, "CUDA:", CUDA_VERSION)

!pip install -q torch_geometric

!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION.replace('.', '')}.html


import torch
import os
import json
import glob
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# 1. Configuration de l'installation
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda

if CUDA_VERSION is None:
    url = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+cpu.html"
else:
    cuda_str = f"cu{CUDA_VERSION.replace('.', '')}"
    url = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{cuda_str}.html"

print(f"Installing dependencies from: {url}")
!pip install -q torch_geometric
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f {url}

def load_graph_data(json_filepath):
    with open(json_filepath, 'r') as f:
        graph_data = json.load(f)

    num_nodes = graph_data['metadata']['nodes_count']
    edges = graph_data['subgraph_topology']['edge_index']
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(graph_data['subgraph_topology']['edge_type_indices'], dtype=torch.long)

    node_registry = graph_data['node_registry']
    # Correction : Accès sécurisé aux features via str(i) car node_registry est un dict
    x_features = []
    for i in range(num_nodes):
        node_key = str(i)
        if node_key in node_registry:
            x_features.append(node_registry[node_key]['features_vector'])
        else:
            # Padding si un noeud manque
            x_features.append([0.0] * 6)

    x = torch.tensor(x_features, dtype=torch.float)
    y = torch.tensor(graph_data['ml_targets']['y_best_alloc'], dtype=torch.float).unsqueeze(1)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

def create_dataset_loader(directory_path, batch_size=32, shuffle=True):
    data_list = []
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)

    search_pattern = os.path.join(directory_path, "*.json")
    json_files = glob.glob(search_pattern)

    print(f"Found {len(json_files)} graph files in '{directory_path}'.")

    for filepath in json_files:
        try:
            data_list.append(load_graph_data(filepath))
        except Exception as e:
            print(f"Error loading {filepath}: {e}")

    if len(data_list) == 0:
        print("WARNING: No data found. DataLoader cannot be created.")
        return None

    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

dataset_folder = './datasets'
loader = create_dataset_loader(dataset_folder, batch_size=4, shuffle=True)

if loader:
    for step, batch in enumerate(loader):
        print(f"Successfully loaded batch with {batch.num_graphs} graphs.")
        break

Torch: 2.10.0 CUDA: None
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION.replace('.', '')}.html'
Installing dependencies from: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.2/828.2 kB 34.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.9/306.9 kB 21.9 MB/s eta 0:00:00
Found 0 graph files in './datasets'.


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx

# Initialize lists to store our metrics
num_nodes_list = []
num_edges_list = []
mean_depths = []
max_depths = []

print("Analyzing directed dataset topologies... this might take a moment.")

# Iterate through the entire dataset to gather stats
for data in loader.dataset:
    num_nodes_list.append(data.num_nodes)
    num_edges_list.append(data.num_edges)

    # Convert to NetworkX, keeping the graph DIRECTED
    G = to_networkx(data, to_undirected=False)

    # FIX 1: Remove self-loops (nodes pointing to themselves) which break source/target detection
    G.remove_edges_from(nx.selfloop_edges(G))

    sources = [n for n, d in G.in_degree() if d == 0]
    targets = [n for n, d in G.out_degree() if d == 0]

    graph_depths = []

    # Strategy A: Try finding paths between pure Sources and Targets
    if len(sources) > 0 and len(targets) > 0:
        for s in sources:
            for t in targets:
                if s != t and nx.has_path(G, s, t):
                    graph_depths.append(nx.shortest_path_length(G, s, t))

    # FIX 2: Fallback if the graph is cyclic (no strict sources/targets)
    # Measure the directed shortest path between ALL pairs of reachable nodes
    if len(graph_depths) == 0:
        for s, lengths in nx.shortest_path_length(G):
            for t, length in lengths.items():
                if s != t and length > 0:
                    graph_depths.append(length)

    # Store the mean and max depth for this specific graph
    if graph_depths:
        mean_depths.append(np.mean(graph_depths))
        max_depths.append(np.max(graph_depths))

# Convert to numpy arrays for easy math
nodes_arr = np.array(num_nodes_list)
edges_arr = np.array(num_edges_list)
mean_depths_arr = np.array(mean_depths)
max_depths_arr = np.array(max_depths)

# --- 1. Print Summary Statistics ---
print("\n=== Dataset Topology Statistics ===")
print(f"Nodes per graph : Mean = {nodes_arr.mean():.2f} ± {nodes_arr.std():.2f}")
print(f"Edges per graph : Mean = {edges_arr.mean():.2f} ± {edges_arr.std():.2f}")
if len(mean_depths_arr) > 0:
    print(f"Directed Graph Mean Depth: {mean_depths_arr.mean():.2f} ± {mean_depths_arr.std():.2f}")
    print(f"Directed Graph Max Depth : {max_depths_arr.mean():.2f} ± {max_depths_arr.std():.2f}")
else:
    print("Notice: Still could not find any directed paths (graphs might be completely disconnected).")

# --- 2. Plot Distributions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dataset Diversity: Directed Topological Distributions', fontsize=16)

# Plot Node Distribution
axes[0].hist(nodes_arr, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Nodes')
axes[0].set_xlabel('Number of Nodes')
axes[0].set_ylabel('Frequency')

# Plot Edge Distribution
axes[1].hist(edges_arr, bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Edges')
axes[1].set_xlabel('Number of Edges')

# Plot Depth Distribution (Mean and Max on the same plot)
if len(mean_depths_arr) > 0:
    axes[2].hist(mean_depths_arr, bins=15, color='mediumpurple', edgecolor='black', alpha=0.7, label='Mean Depth')
    axes[2].hist(max_depths_arr, bins=15, color='salmon', edgecolor='black', alpha=0.5, label='Max Depth')
    axes[2].set_title('Directed Flow Depths')
    axes[2].set_xlabel('Path Length (Number of Edges)')
    axes[2].legend()

plt.tight_layout()
plt.show()

Analyzing directed dataset topologies... this might take a moment.


AttributeError: 'NoneType' object has no attribute 'dataset'

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool # Import pooling layer

# Define a custom GCN model for graph-level prediction
class GraphGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_layers, out_channels, dropout):
        super(GraphGCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.lin = torch.nn.Linear(hidden_channels, out_channels) # Final linear layer after pooling
        self.dropout = dropout

    def forward(self, x, edge_index, batch_map):
        # Apply GCNConv layers
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Global pooling layer to aggregate node features to a single graph feature vector
        # 'batch_map' is the tensor that maps each node to its respective graph in the batch
        x = global_mean_pool(x, batch_map)

        # Apply final linear layer for prediction
        x = self.lin(x)
        return x

# 1. Initialize the custom GraphGCN model
# - in_channels: 6 (matching the length of features_vector)
# - hidden_channels: 16 (number of hidden units, adjustable)
# - num_layers: 2 (number of GCNConv layers)
# - out_channels: 1 (predicting the single y_best_alloc value)
model = GraphGCN(
    in_channels=6,
    hidden_channels=16,
    num_layers=2,
    out_channels=1,
    dropout=0.5
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        # Pass the 'batch.batch' tensor (which maps nodes to graphs) to the forward method for pooling
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = F.mse_loss(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

for epoch in range(1, 101):
    loss = train(loader)
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

/tmp/ipykernel_1179/1072626183.py:53: UserWarning: Using a target size (torch.Size([92, 1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(out, batch.y)


Epoch: 010, Loss: 0.0144
Epoch: 020, Loss: 0.0071
Epoch: 030, Loss: 0.0059
Epoch: 040, Loss: 0.0060
Epoch: 050, Loss: 0.0065
Epoch: 060, Loss: 0.0061
Epoch: 070, Loss: 0.0064
Epoch: 080, Loss: 0.0060
Epoch: 090, Loss: 0.0095
Epoch: 100, Loss: 0.0059


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn.models import GAT

# 1. Initialize the pre-built GAT model
gat_model = GAT(
    in_channels=6,        # Matches your 6 node features
    hidden_channels=16,
    num_layers=2,
    out_channels=1,       # Predicting y_best_alloc
    heads=2,              # Uses 2 attention heads per layer
    dropout=0.5
)

gat_optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.01)

def train_gat(loader):
    gat_model.train()
    total_loss = 0
    for batch in loader:
        gat_optimizer.zero_grad()
        out = gat_model(batch.x, batch.edge_index)
        loss = F.mse_loss(out, batch.y)
        loss.backward()
        gat_optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

print("Starting GAT Training...")
for epoch in range(1, 101):
    loss = train_gat(loader)
    if epoch % 10 == 0:
        print(f'GAT Epoch: {epoch:03d}, Loss: {loss:.4f}')

Starting GAT Training...


/tmp/ipykernel_1179/4068629970.py:23: UserWarning: Using a target size (torch.Size([92, 1])) that is different to the input size (torch.Size([97, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(out, batch.y)


RuntimeError: The size of tensor a (97) must match the size of tensor b (92) at non-singleton dimension 0